# Question Answering with LangChain, OpenAI, and MultiQuery Retriever

This interactive workbook demonstrates example of Elasticsearch's [MultiQuery Retriever](https://api.python.langchain.com/en/latest/retrievers/langchain.retrievers.multi_query.MultiQueryRetriever.html) to generate similar queries for a given user input and apply all queries to retrieve a larger set of relevant documents from a vectorstore.

Before we begin, we first split the fictional workplace documents into passages with `langchain` and uses OpenAI to transform these passages into embeddings and then store these into Elasticsearch.

We will then ask a question, generate similar questions using langchain and OpenAI, retrieve relevant passages from the vector store, and use langchain and OpenAI again to provide a summary for the questions.

## Install packages and import modules

In [14]:
!python3 -m pip install -qU jq lark langchain langchain-elasticsearch langchain_openai langchain-community langchain-text-splitters tiktoken

from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_elasticsearch import ElasticsearchStore
from langchain_openai.llms import OpenAI
from langchain_classic.retrievers.multi_query import MultiQueryRetriever
from getpass import getpass

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-classic 1.0.0 requires langchain-core<2.0.0,>=1.0.0, but you have langchain-core 0.3.80 which is incompatible.
langchain-classic 1.0.0 requires langchain-text-splitters<2.0.0,>=1.0.0, but you have langchain-text-splitters 0.3.11 which is incompatible.
langchain-experimental 0.4.0 requires langchain-community<1.0.0,>=0.4.0, but you have langchain-community 0.3.31 which is incompatible.
langchain-experimental 0.4.0 requires langchain-core<2.0.0,>=1.0.0, but you have langchain-core 0.3.80 which is incompatible.
langgraph-prebuilt 1.0.4 requires langchain-core>=1.0.0, but you have langchain-core 0.3.80 which is incompatible.


## Connect to Elasticsearch

ℹ️ We're using an Elastic Cloud deployment of Elasticsearch for this notebook. If you don't have an Elastic Cloud deployment, sign up [here](https://cloud.elastic.co/registration?utm_source=github&utm_content=elasticsearch-labs-notebook) for a free trial. 

We'll use the **Cloud ID** to identify our deployment, because we are using Elastic Cloud deployment. To find the Cloud ID for your deployment, go to https://cloud.elastic.co/deployments and select your deployment.

We will use [ElasticsearchStore](https://api.python.langchain.com/en/latest/vectorstores/langchain.vectorstores.elasticsearch.ElasticsearchStore.html) to connect to our elastic cloud deployment, This would help create and index data easily.  We would also send list of documents that we created in the previous step

In [3]:
# https://www.elastic.co/search-labs/tutorials/install-elasticsearch/elastic-cloud#finding-your-cloud-id
ELASTIC_CLOUD_ID = getpass("Elastic Cloud ID: ")

# https://www.elastic.co/search-labs/tutorials/install-elasticsearch/elastic-cloud#creating-an-api-key
ELASTIC_API_KEY = getpass("Elastic Api Key: ")

# https://platform.openai.com/api-keys
OPENAI_API_KEY = getpass("OpenAI API key: ")

embeddings = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY)

vectorstore = ElasticsearchStore(
    es_cloud_id=ELASTIC_CLOUD_ID,
    es_api_key=ELASTIC_API_KEY,
    index_name='test', #give it a meaningful name,
    embedding=embeddings,
)

## Indexing Data into Elasticsearch
Let's download the sample dataset and deserialize the document.

In [4]:
from urllib.request import urlopen
import json

url = "https://raw.githubusercontent.com/elastic/elasticsearch-labs/main/example-apps/chatbot-rag-app/data/data.json"

response = urlopen(url)
data = json.load(response)

with open("temp.json", "w") as json_file:
    json.dump(data, json_file)

### Split Documents into Passages

We’ll chunk documents into passages in order to improve the retrieval specificity and to ensure that we can provide multiple passages within the context window of the final question answering prompt.

Here we are chunking documents into 800 token passages with an overlap of 400 tokens.

Here we are using a simple splitter but Langchain offers more advanced splitters to reduce the chance of context being lost.

In [15]:
from langchain.document_loaders import JSONLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter


def metadata_func(record: dict, metadata: dict) -> dict:
    #Populate the metadata dictionary with keys name, summary, url, category, and updated_at.
    metadata["name"] = record.get("name")
    metadata["summary"] = record.get("summary")
    metadata["url"] = record.get("url")
    metadata["category"] = record.get("category")
    metadata["updated_at"] = record.get("updated_at")

    return metadata


# For more loaders https://python.langchain.com/docs/modules/data_connection/document_loaders/
# And 3rd party loaders https://python.langchain.com/docs/modules/data_connection/document_loaders/#third-party-loaders
loader = JSONLoader(
    file_path="temp.json",
    jq_schema=".[]",
    content_key="content",
    metadata_func=metadata_func,
)

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=800, chunk_overlap=400 #define chunk size and chunk overlap
)
docs = loader.load_and_split(text_splitter=text_splitter)

### Bulk Import Passages

Now that we have split each document into the chunk size of 800, we will now index data to elasticsearch using [ElasticsearchStore.from_documents](https://api.python.langchain.com/en/latest/vectorstores/langchain.vectorstores.elasticsearch.ElasticsearchStore.html#langchain.vectorstores.elasticsearch.ElasticsearchStore.from_documents).

We will use Cloud ID, Password and Index name values set in the `Create cloud deployment` step.

In [16]:
documents = vectorstore.from_documents(
    docs,
    embeddings,
    index_name='test',
    es_cloud_id=ELASTIC_CLOUD_ID,
    es_api_key=ELASTIC_API_KEY,
)

llm = OpenAI(temperature=0, openai_api_key=OPENAI_API_KEY)

retriever = MultiQueryRetriever.from_llm(vectorstore.as_retriever(), llm)

# Question Answering with MultiQuery Retriever

Now that we have the passages stored in Elasticsearch, we can now ask a question to get the relevant passages.

In [18]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain.schema import format_document

import logging

logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

LLM_CONTEXT_PROMPT = ChatPromptTemplate.from_template(
    """You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Be as verbose and educational in your response as possible. 
    
    context: {context}
    Question: "{question}"
    Answer:
    """
)

LLM_DOCUMENT_PROMPT = PromptTemplate.from_template(
    """
---
SOURCE: {name}
{page_content}
---
"""
)


def _combine_documents(
    docs, document_prompt=LLM_DOCUMENT_PROMPT, document_separator="\n\n"
):
    doc_strings = [format_document(doc, document_prompt) for doc in docs]
    return document_separator.join(doc_strings)


_context = RunnableParallel(
    context=retriever | _combine_documents,
    question=RunnablePassthrough(),
)

chain = _context | LLM_CONTEXT_PROMPT | llm

ans = chain.invoke("what is the nasa sales team?")

print("---- Answer ----")
print(ans)

---- Answer ----
The NASA sales team is a part of the Americas region in the sales organization of the company. It is led by two Area Vice-Presidents, Laura Martinez for North America and Gary Johnson for South America. The team is responsible for promoting and selling the company's products and services in the North and South American markets. They work closely with other departments, such as marketing, product development, and customer support, to ensure the company's success in these regions.


**Generate at least two new iteratioins of the previous cells - Be creative.** Did you master Multi-
Query Retriever concepts through this lab?

## Iteration 1: Policy Change Timeline Analyzer

This iteration demonstrates using MultiQuery Retriever to track how workplace policies have evolved over time, particularly useful for understanding policy updates.

In [24]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain.schema import format_document

# Timeline-focused prompt that emphasizes dates and policy evolution
TIMELINE_PROMPT = ChatPromptTemplate.from_template(
    """You are a policy analyst specializing in tracking organizational changes over time.
    
    Analyze the provided documents and answer the question with special attention to:
    1. Timeline of changes (when policies were created or updated)
    2. Evolution of policies (how things have changed over time)
    3. Current state vs. historical context
    4. Any patterns or trends in policy updates
    
    When discussing dates, always use this format: Month Year (e.g., March 2020, May 2023)
    If multiple versions of a policy exist, explain the progression clearly.
    
    Context:
    {context}
    
    Question: {question}
    
    Timeline Analysis:
    """
)

CHRONOLOGICAL_DOC_PROMPT = PromptTemplate.from_template(
    """
┌─────────────────────────────────────────┐
│ 📄 {name}
│ 🔄 Updated: {updated_at}
│ 🏷️  Category: {category}
└─────────────────────────────────────────┘

{page_content}

"""
)

def _combine_docs_chronologically(docs, document_prompt=CHRONOLOGICAL_DOC_PROMPT, document_separator="\n"):
    """Combine documents and sort by date for timeline analysis"""
    # Sort documents by updated_at date (most recent first)
    try:
        sorted_docs = sorted(docs, key=lambda d: d.metadata.get('updated_at', '1900-01-01'), reverse=True)
    except:
        sorted_docs = docs
    
    doc_strings = [format_document(doc, document_prompt) for doc in sorted_docs]
    return document_separator.join(doc_strings)

# Build timeline analysis chain
timeline_context = RunnableParallel(
    context=retriever | _combine_docs_chronologically,
    question=RunnablePassthrough(),
)

timeline_chain = timeline_context | TIMELINE_PROMPT | llm

# Ask questions about policy evolution
print("=" * 70)
print("QUERY 1: Policy Evolution Over Time")
print("=" * 70)
timeline_answer1 = timeline_chain.invoke(
    "How has the work from home policy changed since 2020?"
)
print(timeline_answer1)
print("\n")

print("=" * 70)
print("QUERY 2: Current vs Historical State")
print("=" * 70)
timeline_answer2 = timeline_chain.invoke(
    "What is the current remote work requirement and how does it differ from the original policy?"
)
print(timeline_answer2)
print("\n")

QUERY 1: Policy Evolution Over Time
- March 2020: The original work from home policy was created in response to the COVID-19 pandemic.
    - May 2022: A new update was made to the policy, requiring employees to work from the office two days a week.
    - April 2023: Another update was made, changing the requirement to three days in the office and two days remote work.

Evolution of Policies:
The work from home policy has evolved from a full-time remote work arrangement to a hybrid model, with employees required to work from the office for a certain number of days per week. This change reflects the company's prioritization of employee well-being and the gradual return to in-person work as the pandemic situation improves.

Current State vs. Historical Context:
The current state of the work from home policy is a hybrid model, with employees working from the office for a portion of the week. This is a significant change from the original policy, which allowed for full-time remote work. Thi

## Iteration 2: Role-Based HR Assistant

This iteration simulates an HR assistant that provides personalized responses based on employee roles and departments, with context-aware filtering.

In [22]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain.schema import format_document

# Role-based HR assistant prompt
HR_ASSISTANT_PROMPT = ChatPromptTemplate.from_template(
    """You are an experienced HR assistant helping employees understand company policies and procedures.
    
    Your role is to:
    - Provide clear, actionable guidance based on official company documentation
    - Reference specific policies and their URLs when applicable
    - Highlight any role-specific or department-specific considerations
    - Suggest next steps or who to contact for additional help
    - Be empathetic and supportive in your tone
    
    Employee Role: {employee_role}
    Department: {department}
    
    Relevant Policies & Information:
    {context}
    
    Employee Question: {question}
    
    HR Assistant Response:
    """
)

HR_DOC_PROMPT = PromptTemplate.from_template(
    """
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📋 Policy: {name}
🔗 Reference: {url}
📅 Last Updated: {updated_at}
🏢 Category: {category}
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Summary: {summary}

Details:
{page_content}
"""
)

def _combine_docs_with_metadata(docs, document_prompt=HR_DOC_PROMPT, document_separator="\n\n"):
    """Combine documents with full metadata for HR context"""
    doc_strings = [format_document(doc, document_prompt) for doc in docs[:4]]  # Top 4 most relevant
    return document_separator.join(doc_strings)

def _add_employee_context(input_dict):
    """Add employee role and department context to the query"""
    return {
        "context": input_dict.get("context", ""),
        "question": input_dict.get("question", ""),
        "employee_role": input_dict.get("employee_role", "Employee"),
        "department": input_dict.get("department", "General")
    }

# Build HR assistant chain with employee context
hr_base_context = RunnableParallel(
    context=lambda x: x["question"],
    question=lambda x: x["question"],
    employee_role=lambda x: x.get("employee_role", "Employee"),
    department=lambda x: x.get("department", "General")
)

hr_retrieval = RunnableParallel(
    context=lambda x: retriever.invoke(x["context"]),
    question=lambda x: x["question"],
    employee_role=lambda x: x["employee_role"],
    department=lambda x: x["department"]
)

hr_formatting = RunnableLambda(lambda x: {
    "context": _combine_docs_with_metadata(x["context"]),
    "question": x["question"],
    "employee_role": x["employee_role"],
    "department": x["department"]
})

hr_chain = hr_base_context | hr_retrieval | hr_formatting | HR_ASSISTANT_PROMPT | llm

# Simulate different employee inquiries
print("╔" + "=" * 68 + "╗")
print("║" + " SCENARIO 1: New Engineer - Onboarding Question ".center(68) + "║")
print("╚" + "=" * 68 + "╝")
hr_answer1 = hr_chain.invoke({
    "question": "I'm starting next week. What do I need to do on my first day?",
    "employee_role": "Software Engineer",
    "department": "Engineering"
})
print(hr_answer1)
print("\n")

print("╔" + "=" * 68 + "╗")
print("║" + " SCENARIO 2: Sales Manager - Benefits Question ".center(68) + "║")
print("╚" + "=" * 68 + "╝")
hr_answer2 = hr_chain.invoke({
    "question": "What vacation time am I eligible for and how do I request it?",
    "employee_role": "Area Vice-President",
    "department": "Sales"
})
print(hr_answer2)
print("\n")

print("╔" + "=" * 68 + "╗")
print("║" + " SCENARIO 3: Remote Employee - Policy Question ".center(68) + "║")
print("╚" + "=" * 68 + "╝")
hr_answer3 = hr_chain.invoke({
    "question": "How many days per week do I need to be in the office?",
    "employee_role": "Product Manager",
    "department": "Product"
})
print(hr_answer3)
print("\n")

╔====================================================================╗
║           SCENARIO 1: New Engineer - Onboarding Question           ║
╚====================================================================╝
Welcome to our team! On your first day, please attend the orientation session to meet your colleagues and learn more about our company's culture and values. You will also receive a copy of our employee handbook, which contains important information about our policies and procedures. Please review it carefully and reach out to the HR department if you have any questions. Additionally, you will need to complete any required training sessions and update your tax elections and benefits enrollment forms. Your supervisor and the HR department will guide you through these steps. We look forward to having you on board!


╔====================================================================╗
║           SCENARIO 2: Sales Manager - Benefits Question            ║
╚=======================